In [ ]:
from google.colab import drive
import torch
import sys
import numpy as np
import math
import time
import os
import torch.nn.functional as F
from transformers import get_cosine_schedule_with_warmup
drive.mount('/content/gdrive', force_remount=True)


base_dir = "/content/gdrive/MyDrive/Junior/Second Semester/CS 6787 - Advanced ML Systems/regen-gpt2"
sys.path.append(base_dir)

ckpt_path = os.path.join(base_dir, "Checkpoints/ckpt_baseline_step_3623.pt")
from Models.Baseline_Model.GPT2_Baseline import GPT2_Baseline
from Models.Baseline_Model.test_Baseline import evaluate_statistical_matrics, evaluate_system_metrics
from Models.Configs import TrainConfig, BaselineConfig
from Datasets.DataLoader import CombinedBinDataLoader

device = torch.device("cuda" if torch.cuda.is_available()
                else ("mps" if torch.backends.mps.is_available()
                else "cpu"))
print(F"Device set to {device}")



#Load the trained Model
checkpoint = torch.load(ckpt_path)

model_config = BaselineConfig()
train_config = TrainConfig()


model = GPT2_Baseline(model_config, device).to(device)
model = torch.compile(model)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

Mounted at /content/gdrive
/bin/bash: line 1: get_pwd: command not found


ModuleNotFoundError: No module named 'Models.Baseline_Model.test_Baseline'

In [ ]:
#Load the combined_bin dataset from drive and place it in Colab's files
!cp "/content/gdrive/MyDrive/Junior/Second Semester/CS 6787 - Advanced ML Systems/regen-gpt2/Datasets/fineweb_1B.bin" "/content/fineweb_1B.bin"
_, val_loader = CombinedBinDataLoader.create_loaders(
    '/content/fineweb_1B.bin',
    train_config.batch_per_iter,
    model_config.block_size,
    train_config,
    seed=42
)

Initialized loader with 38,655 chunks of size 24577.
Initialized loader with 2,035 chunks of size 24577.


In [ ]:
# def evaluate_statistical_matrics(model, val_loader, device, max_batches):
#     model.eval()

#     total_correct = 0
#     total_tokens = 0
#     total_loss = 0.0
#     num_batches = 0

#     with torch.no_grad():
#         for x, y in val_loader:
#             #val_loader can grab data infinitely so max_batches should be set
#             if num_batches >= max_batches:
#                 break

#             x, y = x.to(device), y.to(device)
#             logits, loss = model(x, y)

#             #Track accuracy by checking the token with the highest probability
#             preds = logits.argmax(dim=-1)

#             #Ignore masks (not that there should be(?))
#             mask = y != model.config.pad_token_id

#             total_correct += (preds[mask] == y[mask]).sum().item()
#             total_tokens +=  mask.sum().item()
#             total_loss += loss.item()
#             num_batches += 1

#     accuracy = total_correct / total_tokens if total_tokens > 0 else 0.0
#     avg_loss = total_loss / num_batches if num_batches > 0 else 0.0

#     return {
#         "accuracy": accuracy,
#         "avg_loss": avg_loss,
#         "PPL": torch.exp(torch.tensor(avg_loss)).item(),
#         "tokens_seen": total_tokens,
#     }


W0427 03:58:03.043000 23282 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode


{'accuracy': 0.19240573899082755,
 'avg_loss': 5.4503175950050355,
 'PPL': 232.83204650878906,
 'tokens_seen': 4908180}

In [ ]:
# def evaluate_system_metrics(model, device, prompt,
#                                     gen_lengths=[32, 64, 128, 256, 512], num_runs=5):
#     results = {}

#     for max_new_tokens in gen_lengths:
#         latencies = []

#         #Warmup the GPU incase
#         for _ in range(2):
#             model.infer(prompt, max_new_tokens=max_new_tokens)

#         if device == "cuda":
#             torch.cuda.synchronize()
#         for _ in range(num_runs):
#             if device == "cuda":
#                 torch.cuda.synchronize()
#             start = time.perf_counter()
#             model.infer(prompt, max_new_tokens=max_new_tokens)
#             if device == "cuda":
#                 torch.cuda.synchronize()
#             end = time.perf_counter()
#             latencies.append(end - start)

#         avg_latency = sum(latencies) / len(latencies)
#         results[max_new_tokens] = {
#             "throughput_tokens_per_sec": max_new_tokens / avg_latency,
#             "avg_latency_ms": avg_latency * 1000,
#             "ms_per_token": (avg_latency * 1000) / max_new_tokens,
#         }

#     return results


{32: {'throughput_tokens_per_sec': 118.01047746878008,
  'avg_latency_ms': 271.1623636000089,
  'ms_per_token': 8.473823862500279},
 64: {'throughput_tokens_per_sec': 122.79151088118667,
  'avg_latency_ms': 521.2086694000087,
  'ms_per_token': 8.143885459375136},
 128: {'throughput_tokens_per_sec': 122.45819247386731,
  'avg_latency_ms': 1045.2546899001088,
  'ms_per_token': 8.1660522648446},
 256: {'throughput_tokens_per_sec': 127.53461159532677,
  'avg_latency_ms': 2007.2982290666305,
  'ms_per_token': 7.841008707291525},
 512: {'throughput_tokens_per_sec': 126.57909327539505,
  'avg_latency_ms': 4044.901782366651,
  'ms_per_token': 7.900198793684865}}

In [ ]:
evaluate_statistical_matrics(model, val_loader, device, max_batches=200)
evaluate_system_metrics(model, device, "A", num_runs=30)

NameError: name 'evaluate_statistical_matrics' is not defined